# 01 — Data collection: 2025-26 tier-1 pro season from OpenDota

Downloads detailed stats for every match of the tier-1 circuit since TI 2025
(leagues matched by name pattern — see `src/collect_data.py:TIER1_PATTERNS`).

**Requirements:** network access to `api.opendota.com` (keyless ≈60 req/min;
set `OPENDOTA_API_KEY` to go faster). First full pull ≈ 30–90 min; everything
is disk-cached under `data/cache/`, so re-runs are instant and offline-safe.
If this sandbox blocks OpenDota, run this notebook locally once — the other
notebooks only need the parquet files it produces.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
# Pull (or refresh) the season. Tune --since / --max-matches as needed.
from src.collect_data import main as collect
import sys
sys.argv = ['collect', '--since', '2025-09-01']
try:
    collect()
except Exception as e:
    print(f'Collection failed ({e}).\nFalling back to any existing parquet in data/.')

In [ ]:
matches = pd.read_parquet(DATA / 'matches.parquet')
players = pd.read_parquet(DATA / 'match_players.parquet')
picks   = pd.read_parquet(DATA / 'picks_bans.parquet')
print(f'{len(matches):,} matches | {len(players):,} player rows | {len(picks):,} draft rows')
matches['date'] = pd.to_datetime(matches.start_time, unit='s')
matches.groupby('league_name').agg(n=('match_id','size'), first=('date','min'), last=('date','max')).sort_values('last')

In [ ]:
# Sanity: patch coverage and parsed-replay coverage (gold_adv present)
print(matches.patch.value_counts().sort_index())
print(f"parsed replays: {matches.gold_adv_10.notna().mean():.0%}")
matches.sample(3)